In [1]:
!pip install kagglehub tensorflow opencv-python scikit-learn pandas matplotlib -q

In [2]:
import kagglehub
import os
import numpy as np
import pandas as pd
import cv2

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate, Flatten

In [3]:
path = kagglehub.dataset_download("ted8080/house-prices-and-images-socal")
print(path)

100%|██████████| 369M/369M [00:04<00:00, 78.5MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/ted8080/house-prices-and-images-socal/versions/1


In [4]:
csv_file = None

for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(".csv"):
            csv_file = os.path.join(root, f)

print("CSV:", csv_file)

df = pd.read_csv(csv_file)
df.head()

CSV: /root/.cache/kagglehub/datasets/ted8080/house-prices-and-images-socal/versions/1/socal2.csv


,image_id,street,citi,n_citi,bed,bath,sqft,price
0,0,1317 Van Buren Avenue,"Salton City, CA",317,3,2.0,1560,201900
1,1,124 C Street W,"Brawley, CA",48,3,2.0,713,228500
2,2,2304 Clark Road,"Imperial, CA",152,3,1.0,800,273950
3,3,755 Brawley Avenue,"Brawley, CA",48,3,1.0,1082,350000
4,4,2207 R Carrillo Court,"Calexico, CA",55,4,3.0,2547,385100


In [5]:
image_dir = None

for root, dirs, files in os.walk(path):
    img_count = len([f for f in files if f.endswith(('.jpg','.png','.jpeg'))])

    if img_count > 0:
        image_dir = root
        break

print("Image Dir:", image_dir)

Image Dir: /root/.cache/kagglehub/datasets/ted8080/house-prices-and-images-socal/versions/1/socal2/socal_pics


In [6]:
images = []
limit = 1000   # 🔥 IMPORTANT: prevents crash

files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg','.png','.jpeg'))][:limit]

for f in files:
    img = cv2.imread(os.path.join(image_dir, f))

    if img is not None:
        img = cv2.resize(img, (128, 128))  # smaller size = stable
        images.append(img)

X_images = np.array(images, dtype=np.float32)

print(X_images.shape)

(1000, 128, 128, 3)


In [7]:
print(df.columns)

# keep only numeric
df = df.select_dtypes(include=['int64','float64'])

df = df.dropna()

df = df.iloc[:len(X_images)]  # match images

X_tab = df.drop(columns=[df.columns[-1]])  # assume last column = price
y = df[df.columns[-1]]

scaler = StandardScaler()
X_tab = scaler.fit_transform(X_tab)

print(X_tab.shape, y.shape)

Index(['image_id', 'street', 'citi', 'n_citi', 'bed', 'bath', 'sqft', 'price'], dtype='object')
(1000, 5) (1000,)


In [8]:
X_images = X_images / 255.0

In [9]:
X_img_train, X_img_test, X_tab_train, X_tab_test, y_train, y_test = train_test_split(
    X_images,
    X_tab,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
img_input = Input(shape=(128,128,3))

x = Flatten()(img_input)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)

img_out = x

In [17]:


img_input = Input(shape=(128,128,3))

x = tf.keras.layers.Conv2D(32, (3,3), activation='relu')(img_input)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(64, (3,3), activation='relu')(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)

img_out = x

In [18]:
combined = Concatenate()([img_out, tab_out])

z = Dense(64, activation='relu')(combined)
z = Dense(32, activation='relu')(z)

output = Dense(1)(z)

In [19]:
model = Model(inputs=[img_input, tab_input], outputs=output)

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 126, 126,  │        896 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 63, 63,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 61, 61,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 30, 30,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 57600)     │          0 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │        384 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 128)       │  7,372,928 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 160)       │          0 │ dense_7[0][0],    │
│ (Concatenate)       │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │     10,304 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 32)        │      2,080 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         33 │ dense_9[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,407,201 (28.26 MB)

 Trainable params: 7,407,201 (28.26 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = model.fit(
    [X_img_train, X_tab_train],
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=16
)

Epoch 1/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 23s 466ms/step - loss: 703672025088.0000 - mae: 614293.6875 - val_loss: 302032453632.0000 - val_mae: 490489.4062
Epoch 2/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 41s 464ms/step - loss: 350811029504.0000 - mae: 512272.3125 - val_loss: 301503873024.0000 - val_mae: 487290.1562
Epoch 3/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 41s 457ms/step - loss: 347050311680.0000 - mae: 509040.7500 - val_loss: 302395752448.0000 - val_mae: 479189.1562
Epoch 4/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 40s 430ms/step - loss: 347139047424.0000 - mae: 504204.0000 - val_loss: 301973143552.0000 - val_mae: 488246.4062
Epoch 5/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 21s 460ms/step - loss: 345318555648.0000 - mae: 508693.2500 - val_loss: 316104081408.0000 - val_mae: 463826.6562


In [21]:
pred = model.predict([X_img_test, X_tab_test])

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 197ms/step


In [22]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 456095.1875
RMSE: 570578.8335646531
